In [2]:
# Cell 1
# =============================================================================
# Load Augmented DataFrames (Nodes A–H) & Base Label Encoder
# =============================================================================

import os
import joblib
import pandas as pd

NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
AUG_DIR = "dataset/normalized/augmented"
MODEL_DIR = "models"

aug_train = {}
aug_test = {}

# Load precomputed node-level augmented datasets
for node in NODES:
    aug_train[node] = joblib.load(os.path.join(AUG_DIR, f"Node_{node}_aug_train.pkl"))
    aug_test[node]  = joblib.load(os.path.join(AUG_DIR, f"Node_{node}_aug_test.pkl"))

# Load baseline label encoder
le = joblib.load(os.path.join(MODEL_DIR, "label_encoder.joblib"))

print(f"Loaded augmented node datasets from: '{AUG_DIR}'")
print(f"Baseline classes: {le.classes_}")
print(f"Example augmented columns (Node A): {aug_train['A'].columns.tolist()}")

Loaded augmented node datasets from: 'dataset/normalized/augmented'
Baseline classes: ['Backdoor' 'none' 'noneX2' 'syn-flood']
Example augmented columns (Node A): ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'data_source', 'prob_Backdoor', 'prob_none', 'prob_noneX2', 'prob_syn-flood', 'agg_shunt_voltage', 'agg_bus_voltage_V', 'agg_current_mA', 'agg_power_mW', 'agg_prob_Backdoor', 'agg_prob_none', 'agg_prob_noneX2', 'agg_prob_syn-flood', 'agg_weight_sum', 'agg_num_contributors']


In [4]:
# Cell 2
# =============================================================================
# Prepare Augmented Data & Define Feature Sets Per Node (WITH BINARY STATE)
# =============================================================================

import pandas as pd

print("=== Node-Level Augmented Dataset Shapes ===")
for node in NODES:
    # Binary encode State: idle -> 0, charging -> 1 (if not already numeric)
    for df in [aug_train[node], aug_test[node]]:
        if df["State"].dtype == object or isinstance(df["State"].iloc[0], str):
            state_mapping = {"idle": 0, "charging": 1}
            df["State"] = df["State"].map(state_mapping).fillna(0).astype(int)

    print(
        f"Node {node}: "
        f"Train Shape = {aug_train[node].shape}, "
        f"Test Shape = {aug_test[node].shape}"
    )

# ----------------------------------------------------------------------------
# Define Feature Sets (Hardware Sensor Metrics, State & Contexts)
# ----------------------------------------------------------------------------
original_numeric = [
    "shunt_voltage",
    "bus_voltage_V",
    "current_mA",
    "power_mW"
]

# Base local features including binary State
local_features = original_numeric + ["State"]

own_prob_cols = [
    f"prob_{cls_name}"
    for cls_name in le.classes_
]

agg_numeric = [
    f"agg_{feature}"
    for feature in original_numeric
]

agg_prob_cols = [
    f"agg_prob_{cls_name}"
    for cls_name in le.classes_
]

# Model 1: Own hardware + State + Aggregated context
feature_cols_hw_plus_agg = (
    local_features
    + agg_numeric
    + agg_prob_cols
)

# Model 2: Full context (Own hardware + State + Own Probs + Agg Context)
feature_cols_full = (
    local_features
    + own_prob_cols
    + agg_numeric
    + agg_prob_cols
)

print(
    f"\nModel 1 Feature Count "
    f"(Own HW + State + Agg Context): {len(feature_cols_hw_plus_agg)}"
)
print(f"Features: {feature_cols_hw_plus_agg}")

print(
    f"\nModel 2 Feature Count "
    f"(Full Context + State): {len(feature_cols_full)}"
)
print(f"Features: {feature_cols_full}")

=== Node-Level Augmented Dataset Shapes ===
Node A: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node B: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node C: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node D: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node E: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node F: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node G: Train Shape = (56000, 21), Test Shape = (24000, 21)
Node H: Train Shape = (56000, 21), Test Shape = (24000, 21)

Model 1 Feature Count (Own HW + State + Agg Context): 13
Features: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'agg_shunt_voltage', 'agg_bus_voltage_V', 'agg_current_mA', 'agg_power_mW', 'agg_prob_Backdoor', 'agg_prob_none', 'agg_prob_noneX2', 'agg_prob_syn-flood']

Model 2 Feature Count (Full Context + State): 17
Features: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'prob_Backdoor', 'prob_none', 'prob_noneX2', 'prob_sy

In [5]:
# Cell 3
# =============================================================================
# Export Node-Specific Augmented Datasets to CSV
# =============================================================================

import os

CSV_OUTPUT_DIR = "dataset/normalized/augmented_csv"
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

for node in NODES:
    train_csv_path = os.path.join(
        CSV_OUTPUT_DIR,
        f"Node_{node}_combined_augmented_train.csv"
    )

    test_csv_path = os.path.join(
        CSV_OUTPUT_DIR,
        f"Node_{node}_combined_augmented_test.csv"
    )

    aug_train[node].to_csv(
        train_csv_path,
        index=False
    )

    aug_test[node].to_csv(
        test_csv_path,
        index=False
    )

    print(f"Saved Node {node} CSV files:")
    print(
        f"  - Train: {train_csv_path} "
        f"(Shape: {aug_train[node].shape})"
    )
    print(
        f"  - Test:  {test_csv_path} "
        f"(Shape: {aug_test[node].shape})"
    )

Saved Node A CSV files:
  - Train: dataset/normalized/augmented_csv/Node_A_combined_augmented_train.csv (Shape: (56000, 21))
  - Test:  dataset/normalized/augmented_csv/Node_A_combined_augmented_test.csv (Shape: (24000, 21))
Saved Node B CSV files:
  - Train: dataset/normalized/augmented_csv/Node_B_combined_augmented_train.csv (Shape: (56000, 21))
  - Test:  dataset/normalized/augmented_csv/Node_B_combined_augmented_test.csv (Shape: (24000, 21))
Saved Node C CSV files:
  - Train: dataset/normalized/augmented_csv/Node_C_combined_augmented_train.csv (Shape: (56000, 21))
  - Test:  dataset/normalized/augmented_csv/Node_C_combined_augmented_test.csv (Shape: (24000, 21))
Saved Node D CSV files:
  - Train: dataset/normalized/augmented_csv/Node_D_combined_augmented_train.csv (Shape: (56000, 21))
  - Test:  dataset/normalized/augmented_csv/Node_D_combined_augmented_test.csv (Shape: (24000, 21))
Saved Node E CSV files:
  - Train: dataset/normalized/augmented_csv/Node_E_combined_augmented_train.

In [7]:
# Cell 4
# =============================================================================
# Model 1: Single Global RF (Own HW & State [2x] + Agg Context [1x])
# Train ONCE on pooled dataset -> Test on each node individually & Loss Values
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

all_classes = le.classes_
num_classes = len(all_classes)
labels_arr = np.arange(num_classes)

# 1. Pool All Training Data & Apply 2:1 Feature Duplication
train_dfs = [aug_train[node].copy() for node in NODES]
df_train_all = pd.concat(train_dfs, ignore_index=True)

own_cols_m1 = [c for c in feature_cols_hw_plus_agg if not c.startswith("agg_")]
X_train_all = df_train_all[feature_cols_hw_plus_agg].copy()
y_train_all = le.transform(df_train_all["Attack"])

for col in own_cols_m1:
    X_train_all[f"{col}_w2"] = X_train_all[col]

numeric_own_m1 = [c for c in own_cols_m1 if c in original_numeric]
numeric_all_m1 = numeric_own_m1 + [f"{c}_w2" for c in numeric_own_m1] + agg_numeric

scaler_m1 = StandardScaler()
X_train_all[numeric_all_m1] = scaler_m1.fit_transform(X_train_all[numeric_all_m1])

# 2. Train the Single Global Random Forest Model
rf_m1_global = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    min_samples_split=20,
    max_samples=0.8,
    random_state=42,
    oob_score=True,
    n_jobs=-1,
)
rf_m1_global.fit(X_train_all.to_numpy(), y_train_all)

prob_train_all = rf_m1_global.predict_proba(X_train_all.to_numpy())
pred_train_all = np.argmax(prob_train_all, axis=1)

print("=== Model 1 Global Training Performance (Pooled Nodes A-H) ===")
print(f"  Train Accuracy:        {accuracy_score(y_train_all, pred_train_all):.6f}")
print(f"  Train Macro F1:        {f1_score(y_train_all, pred_train_all, average='macro'):.6f}")
print(f"  Train Log Loss:        {log_loss(y_train_all, prob_train_all, labels=labels_arr):.6f}")
print(f"  OOB Score:             {rf_m1_global.oob_score_:.6f}\n")

# 3. Evaluate the Trained Model on Each Node Individually
metrics_records_m1 = []
cm_dict_m1 = {}
all_y_true = []
all_y_pred = []
all_prob_m1 = []

for node in NODES:
    test_df = aug_test[node].copy()
    y_test_node = le.transform(test_df["Attack"])
    
    X_test_node = test_df[feature_cols_hw_plus_agg].copy()
    for col in own_cols_m1:
        X_test_node[f"{col}_w2"] = X_test_node[col]
        
    X_test_node[numeric_all_m1] = scaler_m1.transform(X_test_node[numeric_all_m1])
    
    prob_test_node = rf_m1_global.predict_proba(X_test_node.to_numpy())
    pred_test_node = np.argmax(prob_test_node, axis=1)
    
    all_y_true.append(y_test_node)
    all_y_pred.append(pred_test_node)
    all_prob_m1.append(prob_test_node)
    
    macro_f1 = f1_score(y_test_node, pred_test_node, average="macro")
    mse_error = mean_squared_error(y_test_node, pred_test_node)
    test_loss = log_loss(y_test_node, prob_test_node, labels=labels_arr)
    
    try:
        roc_auc = roc_auc_score(
            y_test_node, prob_test_node, multi_class="ovr", average="macro", labels=labels_arr
        )
    except ValueError:
        roc_auc = np.nan
        
    cm = confusion_matrix(y_test_node, pred_test_node, labels=labels_arr)
    cm_dict_m1[node] = cm
    
    metrics_records_m1.append({
        "Node": f"Node {node}",
        "Samples": len(test_df),
        "Test Loss": test_loss,
        "Macro F1": macro_f1,
        "MSE": mse_error,
        "ROC-AUC": roc_auc,
    })
    
    print(f"=== Model 1: Node {node} Test Evaluation (Samples: {len(test_df)}) ===")
    print(f"  Test Loss (Log Loss):  {test_loss:.6f}")
    print(f"  Macro F1-Score:        {macro_f1:.6f}")
    print(f"  Mean Squared Error:    {mse_error:.6f}")
    print(f"  ROC-AUC Score:         {roc_auc:.6f}")
    print(f"  Confusion Matrix:\n  {cm.tolist()}\n")

# Save global Model 1 artifacts
joblib.dump(rf_m1_global, os.path.join(MODELS_DIR, "global_rf_model1_own_hw_plus_agg_state.joblib"))
joblib.dump(scaler_m1, os.path.join(MODELS_DIR, "global_scaler_model1_state.joblib"))

df_summary_m1 = pd.DataFrame(metrics_records_m1)
print("=============================================================================")
print("PER-NODE SUMMARY (GLOBAL MODEL 1 WITH STATE):")
print(df_summary_m1.to_string(index=False))

y_true_all_test = np.concatenate(all_y_true)
y_pred_all_test = np.concatenate(all_y_pred)
y_prob_all_test = np.concatenate(all_prob_m1, axis=0)

print("\n=== Global Combined Test Set Metrics (All Nodes) ===")
global_test_loss_m1 = log_loss(y_true_all_test, y_prob_all_test, labels=labels_arr)
print(f"  Global Test Loss:      {global_test_loss_m1:.6f}")
print(f"  Accuracy:              {accuracy_score(y_true_all_test, y_pred_all_test):.6f}")
print(f"  Macro F1:              {f1_score(y_true_all_test, y_pred_all_test, average='macro'):.6f}")
print(f"  Weighted F1:           {f1_score(y_true_all_test, y_pred_all_test, average='weighted'):.6f}")
print(f"  MSE:                   {mean_squared_error(y_true_all_test, y_pred_all_test):.6f}")
try:
    global_roc_auc = roc_auc_score(
        y_true_all_test, y_prob_all_test, multi_class="ovr", average="macro", labels=labels_arr
    )
    print(f"  ROC-AUC (Macro OVR):   {global_roc_auc:.6f}")
except ValueError:
    pass
print("=============================================================================\n")

# 4. Compute Global Train vs. Test Loss Progression Values over Estimators
print("Computing incremental loss progression values for Model 1...")
node_data_m1 = []
for node in NODES:
    t_df = aug_train[node].copy()
    e_df = aug_test[node].copy()
    
    yt = le.transform(t_df["Attack"])
    ye = le.transform(e_df["Attack"])
    
    Xt = t_df[feature_cols_hw_plus_agg].copy()
    Xe = e_df[feature_cols_hw_plus_agg].copy()
    for col in own_cols_m1:
        Xt[f"{col}_w2"] = Xt[col]
        Xe[f"{col}_w2"] = Xe[col]
    Xt[numeric_all_m1] = scaler_m1.transform(Xt[numeric_all_m1])
    Xe[numeric_all_m1] = scaler_m1.transform(Xe[numeric_all_m1])
    
    node_data_m1.append({
        "X_train": Xt.to_numpy(), "y_train": yt,
        "X_test": Xe.to_numpy(), "y_test": ye,
        "train_accum": np.zeros((len(yt), num_classes)),
        "test_accum": np.zeros((len(ye), num_classes))
    })

y_train_pooled_m1 = np.concatenate([d["y_train"] for d in node_data_m1])
y_test_pooled_m1 = np.concatenate([d["y_test"] for d in node_data_m1])

n_trees_m1 = len(rf_m1_global.estimators_)
m1_train_losses, m1_test_losses = [], []

for t_idx in range(n_trees_m1):
    p_train, p_test = [], []
    for d in node_data_m1:
        tree = rf_m1_global.estimators_[t_idx]
        ptrain_t = tree.predict_proba(d["X_train"])
        ptest_t = tree.predict_proba(d["X_test"])
        
        t_tr_prob = np.zeros((len(d["y_train"]), num_classes))
        t_te_prob = np.zeros((len(d["y_test"]), num_classes))
        for c_idx, c in enumerate(tree.classes_):
            full_idx = np.where(rf_m1_global.classes_ == c)[0][0]
            t_tr_prob[:, full_idx] = ptrain_t[:, c_idx]
            t_te_prob[:, full_idx] = ptest_t[:, c_idx]
            
        d["train_accum"] += t_tr_prob
        d["test_accum"] += t_te_prob
        p_train.append(d["train_accum"] / (t_idx + 1))
        p_test.append(d["test_accum"] / (t_idx + 1))
        
    m1_train_losses.append(log_loss(y_train_pooled_m1, np.concatenate(p_train, axis=0), labels=labels_arr))
    m1_test_losses.append(log_loss(y_test_pooled_m1, np.concatenate(p_test, axis=0), labels=labels_arr))

print("\n=== Model 1 Loss Progression Summary (Start vs. End) ===")
print(f"  Tree 1   -> Train Loss: {m1_train_losses[0]:.6f} | Test Loss: {m1_test_losses[0]:.6f}")
print(f"  Tree 50  -> Train Loss: {m1_train_losses[49]:.6f} | Test Loss: {m1_test_losses[49]:.6f}")
print(f"  Tree 100 -> Train Loss: {m1_train_losses[-1]:.6f} | Test Loss: {m1_test_losses[-1]:.6f}")

=== Model 1 Global Training Performance (Pooled Nodes A-H) ===
  Train Accuracy:        0.953321
  Train Macro F1:        0.960916
  Train Log Loss:        0.124317
  OOB Score:             0.947310



/home/yannic/Desktop/Master_rbg/Semester2/Forschungsarbeit 2/.venvDatasets/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


=== Model 1: Node A Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.200230
  Macro F1-Score:        0.904158
  Mean Squared Error:    0.147833
  ROC-AUC Score:         nan
  Confusion Matrix:
  [[3927, 741, 0, 132], [1016, 13940, 0, 44], [0, 0, 0, 0], [27, 46, 0, 4127]]

=== Model 1: Node B Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.116952
  Macro F1-Score:        0.961153
  Mean Squared Error:    0.061083
  ROC-AUC Score:         0.997797
  Confusion Matrix:
  [[3813, 86, 0, 1], [800, 12931, 0, 69], [0, 0, 1500, 0], [19, 31, 0, 4750]]

=== Model 1: Node C Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.102156
  Macro F1-Score:        0.971068
  Mean Squared Error:    0.050000
  ROC-AUC Score:         0.998161
  Confusion Matrix:
  [[4383, 115, 0, 2], [573, 12257, 0, 70], [0, 0, 1500, 0], [14, 22, 0, 5064]]

=== Model 1: Node D Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.156339
  Macro F1-Score:        0.952605
  

In [8]:
# Cell 5
# =============================================================================
# Model 2: Single Global RF (Full Context: Own HW, State & Probs [2x] + Agg Context [1x])
# Train ONCE on pooled dataset -> Test on each node individually & Loss Values
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

all_classes = le.classes_
num_classes = len(all_classes)
labels_arr = np.arange(num_classes)

# 1. Pool All Training Data & Apply 2:1 Feature Duplication
train_dfs_m2 = [aug_train[node].copy() for node in NODES]
df_train_all_m2 = pd.concat(train_dfs_m2, ignore_index=True)

own_cols_m2 = [c for c in feature_cols_full if not c.startswith("agg_")]
X_train_all_m2 = df_train_all_m2[feature_cols_full].copy()
y_train_all_m2 = le.transform(df_train_all_m2["Attack"])

for col in own_cols_m2:
    X_train_all_m2[f"{col}_w2"] = X_train_all_m2[col]

numeric_own_m2 = [c for c in own_cols_m2 if c in original_numeric]
numeric_all_m2 = numeric_own_m2 + [f"{c}_w2" for c in numeric_own_m2] + [c for c in feature_cols_full if c.startswith("agg_") and c.replace("agg_", "") in original_numeric]

scaler_m2 = StandardScaler()
X_train_all_m2[numeric_all_m2] = scaler_m2.fit_transform(X_train_all_m2[numeric_all_m2])

# 2. Train the Single Global Random Forest Model
rf_m2_global = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    min_samples_split=20,
    max_samples=0.8,
    random_state=42,
    oob_score=True,
    n_jobs=-1,
)
rf_m2_global.fit(X_train_all_m2.to_numpy(), y_train_all_m2)

prob_train_all_m2 = rf_m2_global.predict_proba(X_train_all_m2.to_numpy())
pred_train_all_m2 = np.argmax(prob_train_all_m2, axis=1)

print("=== Model 2 Global Training Performance (Pooled Nodes A-H) ===")
print(f"  Train Accuracy:        {accuracy_score(y_train_all_m2, pred_train_all_m2):.6f}")
print(f"  Train Macro F1:        {f1_score(y_train_all_m2, pred_train_all_m2, average='macro'):.6f}")
print(f"  Train Log Loss:        {log_loss(y_train_all_m2, prob_train_all_m2, labels=labels_arr):.6f}")
print(f"  OOB Score:             {rf_m2_global.oob_score_:.6f}\n")

# 3. Evaluate the Trained Model on Each Node Individually
metrics_records_m2 = []
cm_dict_m2 = {}
all_y_true_m2 = []
all_y_pred_m2 = []
all_prob_m2 = []

for node in NODES:
    test_df = aug_test[node].copy()
    y_test_node = le.transform(test_df["Attack"])
    
    X_test_node = test_df[feature_cols_full].copy()
    for col in own_cols_m2:
        X_test_node[f"{col}_w2"] = X_test_node[col]
        
    X_test_node[numeric_all_m2] = scaler_m2.transform(X_test_node[numeric_all_m2])
    
    prob_test_node = rf_m2_global.predict_proba(X_test_node.to_numpy())
    pred_test_node = np.argmax(prob_test_node, axis=1)
    
    all_y_true_m2.append(y_test_node)
    all_y_pred_m2.append(pred_test_node)
    all_prob_m2.append(prob_test_node)
    
    macro_f1 = f1_score(y_test_node, pred_test_node, average="macro")
    mse_error = mean_squared_error(y_test_node, pred_test_node)
    test_loss = log_loss(y_test_node, prob_test_node, labels=labels_arr)
    
    try:
        roc_auc = roc_auc_score(
            y_test_node, prob_test_node, multi_class="ovr", average="macro", labels=labels_arr
        )
    except ValueError:
        roc_auc = np.nan
        
    cm = confusion_matrix(y_test_node, pred_test_node, labels=labels_arr)
    cm_dict_m2[node] = cm
    
    metrics_records_m2.append({
        "Node": f"Node {node}",
        "Samples": len(test_df),
        "Test Loss": test_loss,
        "Macro F1": macro_f1,
        "MSE": mse_error,
        "ROC-AUC": roc_auc,
    })
    
    print(f"=== Model 2: Node {node} Test Evaluation (Samples: {len(test_df)}) ===")
    print(f"  Test Loss (Log Loss):  {test_loss:.6f}")
    print(f"  Macro F1-Score:        {macro_f1:.6f}")
    print(f"  Mean Squared Error:    {mse_error:.6f}")
    print(f"  ROC-AUC Score:         {roc_auc:.6f}")
    print(f"  Confusion Matrix:\n  {cm.tolist()}\n")

# Save global Model 2 artifacts
joblib.dump(rf_m2_global, os.path.join(MODELS_DIR, "global_rf_model2_full_context_state.joblib"))
joblib.dump(scaler_m2, os.path.join(MODELS_DIR, "global_scaler_model2_state.joblib"))

df_summary_m2 = pd.DataFrame(metrics_records_m2)
print("=============================================================================")
print("PER-NODE SUMMARY (GLOBAL MODEL 2 WITH STATE):")
print(df_summary_m2.to_string(index=False))

y_true_all_test_m2 = np.concatenate(all_y_true_m2)
y_pred_all_test_m2 = np.concatenate(all_y_pred_m2)
y_prob_all_test_m2 = np.concatenate(all_prob_m2, axis=0)

print("\n=== Global Combined Test Set Metrics (All Nodes) ===")
global_test_loss_m2 = log_loss(y_true_all_test_m2, y_prob_all_test_m2, labels=labels_arr)
print(f"  Global Test Loss:      {global_test_loss_m2:.6f}")
print(f"  Accuracy:              {accuracy_score(y_true_all_test_m2, y_pred_all_test_m2):.6f}")
print(f"  Macro F1:              {f1_score(y_true_all_test_m2, y_pred_all_test_m2, average='macro'):.6f}")
print(f"  Weighted F1:           {f1_score(y_true_all_test_m2, y_pred_all_test_m2, average='weighted'):.6f}")
print(f"  MSE:                   {mean_squared_error(y_true_all_test_m2, y_pred_all_test_m2):.6f}")
try:
    global_roc_auc_m2 = roc_auc_score(
        y_true_all_test_m2, y_prob_all_test_m2, multi_class="ovr", average="macro", labels=labels_arr
    )
    print(f"  ROC-AUC (Macro OVR):   {global_roc_auc_m2:.6f}")
except ValueError:
    pass
print("=============================================================================\n")

# 4. Compute Global Train vs. Test Loss Progression Values over Estimators
print("Computing incremental loss progression values for Model 2...")
node_data_m2 = []
for node in NODES:
    t_df = aug_train[node].copy()
    e_df = aug_test[node].copy()
    
    yt = le.transform(t_df["Attack"])
    ye = le.transform(e_df["Attack"])
    
    Xt = t_df[feature_cols_full].copy()
    Xe = e_df[feature_cols_full].copy()
    for col in own_cols_m2:
        Xt[f"{col}_w2"] = Xt[col]
        Xe[f"{col}_w2"] = Xe[col]
    Xt[numeric_all_m2] = scaler_m2.transform(Xt[numeric_all_m2])
    Xe[numeric_all_m2] = scaler_m2.transform(Xe[numeric_all_m2])
    
    node_data_m2.append({
        "X_train": Xt.to_numpy(), "y_train": yt,
        "X_test": Xe.to_numpy(), "y_test": ye,
        "train_accum": np.zeros((len(yt), num_classes)),
        "test_accum": np.zeros((len(ye), num_classes))
    })

y_train_pooled_m2 = np.concatenate([d["y_train"] for d in node_data_m2])
y_test_pooled_m2 = np.concatenate([d["y_test"] for d in node_data_m2])

n_trees_m2 = len(rf_m2_global.estimators_)
m2_train_losses, m2_test_losses = [], []

for t_idx in range(n_trees_m2):
    p_train, p_test = [], []
    for d in node_data_m2:
        tree = rf_m2_global.estimators_[t_idx]
        ptrain_t = tree.predict_proba(d["X_train"])
        ptest_t = tree.predict_proba(d["X_test"])
        
        t_tr_prob = np.zeros((len(d["y_train"]), num_classes))
        t_te_prob = np.zeros((len(d["y_test"]), num_classes))
        for c_idx, c in enumerate(tree.classes_):
            full_idx = np.where(rf_m2_global.classes_ == c)[0][0]
            t_tr_prob[:, full_idx] = ptrain_t[:, c_idx]
            t_te_prob[:, full_idx] = ptest_t[:, c_idx]
            
        d["train_accum"] += t_tr_prob
        d["test_accum"] += t_te_prob
        p_train.append(d["train_accum"] / (t_idx + 1))
        p_test.append(d["test_accum"] / (t_idx + 1))
        
    m2_train_losses.append(log_loss(y_train_pooled_m2, np.concatenate(p_train, axis=0), labels=labels_arr))
    m2_test_losses.append(log_loss(y_test_pooled_m2, np.concatenate(p_test, axis=0), labels=labels_arr))

print("\n=== Model 2 Loss Progression Summary (Start vs. End) ===")
print(f"  Tree 1   -> Train Loss: {m2_train_losses[0]:.6f} | Test Loss: {m2_test_losses[0]:.6f}")
print(f"  Tree 50  -> Train Loss: {m2_train_losses[49]:.6f} | Test Loss: {m2_test_losses[49]:.6f}")
print(f"  Tree 100 -> Train Loss: {m2_train_losses[-1]:.6f} | Test Loss: {m2_test_losses[-1]:.6f}")

=== Model 2 Global Training Performance (Pooled Nodes A-H) ===
  Train Accuracy:        0.974714
  Train Macro F1:        0.978951
  Train Log Loss:        0.066727
  OOB Score:             0.969451



/home/yannic/Desktop/Master_rbg/Semester2/Forschungsarbeit 2/.venvDatasets/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


=== Model 2: Node A Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.124828
  Macro F1-Score:        0.943060
  Mean Squared Error:    0.077500
  ROC-AUC Score:         nan
  Confusion Matrix:
  [[4283, 476, 0, 41], [630, 14362, 0, 8], [0, 0, 0, 0], [29, 23, 0, 4148]]

=== Model 2: Node B Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.074851
  Macro F1-Score:        0.973640
  Mean Squared Error:    0.038750
  ROC-AUC Score:         0.998685
  Confusion Matrix:
  [[3794, 106, 0, 0], [499, 13286, 0, 15], [0, 0, 1500, 0], [21, 19, 0, 4760]]

=== Model 2: Node C Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.067753
  Macro F1-Score:        0.977456
  Mean Squared Error:    0.037417
  ROC-AUC Score:         0.998793
  Confusion Matrix:
  [[4344, 149, 0, 7], [398, 12480, 0, 22], [0, 0, 1500, 0], [16, 14, 0, 5070]]

=== Model 2: Node D Test Evaluation (Samples: 24000) ===
  Test Loss (Log Loss):  0.099259
  Macro F1-Score:        0.971155
  Me